# Create edgotype profiles

In this step, we make one edgotype profile per gene of interactors by alleles. Later, we will explore metrics that summarize overall perturbation for each allele (quantitative score like imaging / abundance)

In [48]:
import polars as pl

In [49]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"

In [50]:
scores = pl.read_csv(f"{outputs_dir}/4_normalized_consensus_scores.csv")
scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
i64,i64,i64,str,f64,i64,i64,i64,f64,f64,str,str,str,str,str,str,str,f64,f64,str
53964,2713,449,"""VUSAPWT1B1""",0.0,1,1,1,1.0,1.0,"""NFIX""","""MLH1""","""Pro28Leu""","""83C>T""","""ENSG00000076242""","""Edgotyping3""","""1_Pathogenic""",6.8405e-7,3.0,"""MLH1_Pro28Leu"""
71393,71191,205704,"""VUSAPWT2B1""",0.375,2,1,1,1.0,1.0,"""SOHLH1""","""SUOX""","""Ala265Asp""","""794C>A""","""ENSG00000139531""","""CEGS2""","""4_VUS""",0.000042,1.0,"""SUOX_Ala265Asp"""
2522,7652,1818,"""VUSAPWT1B1""",0.625,1,1,1,1.0,1.0,"""TRIM38""","""STXBP1""","""Lys333Glu""","""997A>G""","""ENSG00000136854""","""Edgotyping3""","""4_VUS""",6.8406e-7,1.0,"""STXBP1_Lys333Glu"""
3095,71001,205675,"""VUSAPWT2B1""",0.4375,2,1,1,1.0,1.0,"""ZG16B""","""UBQLN2""","""Pro506Thr""","""1516C>A""","""ENSG00000188021""","""CEGS2""","""1_Pathogenic""",null,1.0,"""UBQLN2_Pro506Thr"""
6490,2920,214424,"""VUSAPWT2B1""",0.5625,2,1,1,1.0,1.0,"""PIAS2""","""GFAP""","""Lys236Thr""","""707A>C""","""ENSG00000131095""","""CEGS2""","""5_Others""",null,null,"""GFAP_Lys236Thr"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
6811,4723,201740,"""VUSAPWT1B1""",0.1875,1,1,1,1.0,1.0,"""MAD1L1""","""TPM1""","""Ile172Thr""","""515T>C""","""ENSG00000140416""","""CEGS2""","""4_VUS""",0.000003,2.0,"""TPM1_Ile172Thr"""
11132,52920,2689,"""VUSAPWT1B1""",0.4375,1,1,1,1.0,1.0,"""EPN1""","""LITAF""","""Ser15Leu""","""44C>T""","""ENSG00000189067""","""Edgotyping3""","""3_Conflicting""",null,1.0,"""LITAF_Ser15Leu"""
1969,513,213753,"""VUSAPWT6B1""",0.375,6,1,1,1.0,1.0,"""ARMC12""","""FA2H""","""Arg143Cys""","""427C>T""","""ENSG00000103089""","""CEGS2""","""3_Conflicting""",null,1.0,"""FA2H_Arg143Cys"""


In [51]:
scores_pd = scores.to_pandas()
scores_pd[
    (scores_pd.ad_orf_id == '12626.0') &
    (scores_pd.db_orf_id == '9905.0') #&
    # (scores_pd.db_mut_id == 215290)
    ]

,ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0


In [52]:
scores = pl.read_csv(f"{outputs_dir}/4_normalized_consensus_scores.csv")

# keep only scores where there is at least one matched reference-variant set
orfs_with_alleles = scores.filter(pl.col("db_mut_id") != 0).select("db_orf_id").unique().to_series().to_list()
orfs = scores.filter(pl.col("db_mut_id") == 0).select("db_orf_id").unique().to_series().to_list()

complete_orf_pairs = [i for i in orfs if i in orfs_with_alleles]
scores = scores.filter(pl.col("db_orf_id").is_in(complete_orf_pairs))
genes = scores.select("db_orf_id").to_series().unique().to_list()

# print(scores)

In [53]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    profile = gene_scores.pivot(
                values="normalized_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/manual/orf_id_{gene}.csv")

shape_summary = pl.DataFrame(shape_summary)

In [54]:
## Add a perturbation category for each edge to use for the N2H validation

# Extract WT scores
wt_scores = scores.filter(pl.col("db_mut_id") == 0).select(["db_orf_id", "ad_orf_id", "normalized_score"])

# Merge WT scores back onto the main df
scores_with_wt = scores.join(wt_scores, on=["db_orf_id", "ad_orf_id"], how="left", suffix="_wt")

# Compute allele/WT ratio
scores_with_wt = scores_with_wt.with_columns(
    (pl.col("normalized_score") / pl.col("normalized_score_wt")).alias("ratio")
)

# Determine if the allele is perturbed - binary output
threshold = 0.5
scores_with_wt = scores_with_wt.with_columns(
    ((pl.col("db_mut_id") != 0) & (pl.col("ratio") < threshold)).alias("perturbed")
)

# Build perturbation_classes expression
bins = [
    (1.0, float("inf"), "increased edge"),
    (0.8, 1.0, "similar"),
    (0.6, 0.8, "slightly perturbed"),
    (0.4, 0.6, "perturbed"),
    (0.2, 0.4, "highly perturbed"),
    (0.0, 0.2, "very highly perturbed"),
]

perturbation_expr = pl.when(pl.col("db_mut_id") == 0).then(pl.lit("WT"))

perturbation_expr = perturbation_expr.when(pl.col("normalized_score_wt") == 0.0).then(pl.lit("WT edge lost"))

perturbation_expr = perturbation_expr.when(pl.col("ratio") == 0.0).then(pl.lit("edge lost"))

perturbation_expr = perturbation_expr.when(pl.col("ratio") == 1.0).then(pl.lit("WT-like"))

for lower, upper, label in bins:
    perturbation_expr = perturbation_expr.when((pl.col("ratio") > lower) & (pl.col("ratio") <= upper)).then(pl.lit(label))


perturbation_expr = perturbation_expr.otherwise(pl.lit("unclassified"))




scores_with_wt = scores_with_wt.with_columns(
    perturbation_expr.alias("perturbation_classes")
)

scores_with_perturbed = scores_with_wt.select("*")


pert2 = (pl.when(pl.col("db_mut_id") == 0).then(pl.lit('WT')))

pert2 = pert2.when(pl.col("normalized_score_wt") == 0.0).then(pl.lit("WT edge lost"))

pert2 = pert2.when(pl.col("ratio") == 0.0).then(pl.lit("edge lost"))

pert2 = pert2.when(pl.col("ratio") == 1.0).then(pl.lit("WT-like"))

bins_2 = [
    (float("inf"), 1.0, "increased edge"),
    (1, 0.7, 'slightly perturbed'),
    (0.7, 0.3, 'perturbed'),
    (0.3, 0, 'highly perturbed')
]

for upper, lower, label in bins_2:
    pert2 = pert2.when(
        (pl.col("ratio") <= upper) &
        (pl.col("ratio") >  lower)
    ).then(pl.lit(label))

pert2 = pert2.otherwise(pl.lit("unclassified"))

scores_with_perturbed = scores_with_wt.with_columns(
    perturbation_classes_2 = pert2.alias("perturbation_classes_2")
)



cols_to_keep = ['ad_orf_id',
 'db_orf_id',
 'db_mut_id',
 'normalized_score',
 'ad_symbol',
 'symbol',
 'clinvar_clnsig_clean',
 'normalized_score_wt',
 'ratio',
 'perturbed',
 'perturbation_classes',
 "perturbation_classes_2"]

scores_with_perturbed = scores_with_perturbed.select(cols_to_keep)

scores_with_perturbed.write_csv(f"{outputs_dir}/7_edge_perturbation_categorical.csv")

## Examine profile characteristics

In [55]:
# Number of profiled alleles with more than 5 interactors
hq_orfs = shape_summary.filter(pl.col("n_interactors") >= 5)
num_orfs = hq_orfs.shape[0]
print(num_orfs)

print(hq_orfs.select(pl.col("n_alleles")).sum() - num_orfs)

25
shape: (1, 1)
┌───────────┐
│ n_alleles │
│ ---       │
│ i64       │
╞═══════════╡
│ 255       │
└───────────┘


In [56]:
# Total number of profiled alleles
prof_genes = shape_summary.shape[0]
print(prof_genes)

print(shape_summary.select(pl.col("n_alleles")).sum() - prof_genes)

84
shape: (1, 1)
┌───────────┐
│ n_alleles │
│ ---       │
│ i64       │
╞═══════════╡
│ 659       │
└───────────┘


# Automated

In [57]:
scores = pl.read_csv(f"{outputs_dir}/4_automated_normalized_consensus_scores.csv")
scores = scores.drop_nulls(subset=["normalized_score"])

# keep only scores where there is at least one matched reference-variant set
orfs_with_alleles = scores.filter(pl.col("db_mut_id") != 0).select("db_orf_id").unique().to_series().to_list()
orfs = scores.filter(pl.col("db_mut_id") == 0).select("db_orf_id").unique().to_series().to_list()

complete_orf_pairs = [i for i in orfs if i in orfs_with_alleles]
scores = scores.filter(pl.col("db_orf_id").is_in(complete_orf_pairs))
genes = scores.select("db_orf_id").to_series().unique().to_list()

scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,adjusted_score,wt_summed_score,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
i64,i64,i64,str,f64,f64,f64,i64,i64,i64,f64,f64,str,str,str,str,str,str,str,f64,f64,str
4761,2713,0,"""VUSAPWT1B1""",1362.022266,1362.022266,1.0,1,1,1,null,null,"""AP2B1""","""MLH1""",null,null,"""ENSG00000076242""",null,null,null,null,"""MLH1"""
100009072,70368,205543,"""VUSAPWT1B2""",550.017348,615.652634,0.893389,1,1,1,1.0,1.0,"""PAX6""","""KCTD7""","""Asp115Tyr""","""343G>T""","""ENSG00000243335""","""CEGS2""","""1_Pathogenic""",null,null,"""KCTD7_Asp115Tyr"""
71001,52920,2677,"""VUSAPWT1B1""",1102.398196,2114.542783,0.521341,1,1,1,1.0,1.0,"""UBQLN2""","""LITAF""","""Gly112Ser""","""334G>A""","""ENSG00000189067""","""RC4""","""1_Pathogenic""",null,2.0,"""LITAF_Gly112Ser"""
10064,52920,2677,"""VUSAPWT2B1""",2324.39945,1905.580246,1.219786,2,1,1,1.0,1.0,"""KCNIP4""","""LITAF""","""Gly112Ser""","""334G>A""","""ENSG00000189067""","""RC4""","""1_Pathogenic""",null,2.0,"""LITAF_Gly112Ser"""
100016069,100016069,2630,"""VUSAPWT1B1""",0.0,1503.805617,0.0,1,1,1,1.0,2.0,"""KCNJ2""","""KCNJ2""","""Asn216His""","""646A>C""","""ENSG00000123700""","""Edgotyping3""","""1_Pathogenic""",null,null,"""KCNJ2_Asn216His"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
4519,3449,0,"""VUSAPWT1B2""",940.139564,940.139564,1.0,1,1,1,null,null,"""GMPPA""","""GMPPB""",null,null,"""ENSG00000173540""",null,null,null,null,"""GMPPB"""
11830,7675,202588,"""VUSAPWT2B1""",80.87059,8.111111,9.970347,2,1,1,1.0,1.0,"""KPNA2""","""NMNAT1""","""Arg237Leu""","""710G>T""","""ENSG00000173614""","""CEGS2""","""1_Pathogenic""",null,null,"""NMNAT1_Arg237Leu"""
100070143,3535,6159,"""VUSAPWT1B1""",1806.470986,1462.500818,1.235193,1,1,1,1.0,1.0,"""HSF2BP""","""EMD""","""Ser54Phe""","""161C>T""","""ENSG00000102119""","""RC4""","""1_Pathogenic""",9.1558e-7,1.0,"""EMD_Ser54Phe"""


In [58]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    profile = gene_scores.pivot(
                values="normalized_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/auto_norm/orf_id_{gene}_auto_normalized.csv")

shape_summary = pl.DataFrame(shape_summary)

In [59]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    profile = gene_scores.pivot(
                values="adjusted_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/auto_summed/orf_id_{gene}_auto_summed.csv")

shape_summary = pl.DataFrame(shape_summary)